# SWOW-KEC Statistical Treatment Audit

**Comprehensive audit of SWOW data statistical treatment and KEC metrics integration**

This notebook consolidates validation procedures from scattered scripts into a reproducible format.

In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Configuration
ROOT = Path('.')
DATA_PROC = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
AUDIT_OUT = ROOT / 'notebooks' / 'outputs' / 'audit'
AUDIT_OUT.mkdir(parents=True, exist_ok=True)

print("✅ Environment setup complete")

## 1. Data Loading and Availability Assessment

In [ ]:
# Load datasets
datasets = {}
file_paths = {
    'zuco': DATA_PROC / 'zuco_aligned.csv',
    'kec': DATA_PROC / 'kec' / 'metrics_en.csv',
    'merged': DATA_PROC / 'zuco_kec_merged.csv',
    'coeffs': DATA_PROC / 'models_reading_coeffs.csv'
}

for name, path in file_paths.items():
    if path.exists():
        try:
            datasets[name] = pd.read_csv(path, low_memory=False)
            print(f"✅ {name}: {datasets[name].shape}")
        except Exception as e:
            print(f"❌ {name}: Error loading - {e}")
            datasets[name] = pd.DataFrame()
    else:
        print(f"❌ {name}: File not found")
        datasets[name] = pd.DataFrame()

# Pipeline status
available = [name for name, df in datasets.items() if not df.empty]
pipeline_status = 'COMPLETE' if len(available) >= 3 else 'PARTIAL' if len(available) >= 2 else 'INCOMPLETE'
print(f"\n📊 Pipeline Status: {pipeline_status}")
print(f"📊 Available datasets: {available}")

## 2. KEC Metrics Validation

In [ ]:
# KEC metrics audit
if not datasets['kec'].empty:
    kec_df = datasets['kec']
    
    # Check expected columns
    expected_cols = ['entropy', 'curvature', 'coherence']
    available_cols = [col for col in expected_cols if col in kec_df.columns]
    missing_cols = [col for col in expected_cols if col not in kec_df.columns]
    
    print(f"📊 KEC Metrics Analysis:")
    print(f"  Total nodes: {len(kec_df):,}")
    print(f"  Available metrics: {available_cols}")
    print(f"  Missing metrics: {missing_cols}")
    
    # Statistics for each metric
    kec_stats = {}
    for metric in available_cols:
        values = kec_df[metric].dropna()
        kec_stats[metric] = {
            'count': len(values),
            'missing': kec_df[metric].isna().sum(),
            'mean': values.mean(),
            'std': values.std(),
            'min': values.min(),
            'max': values.max()
        }
        
        # Range validation
        issues = []
        if metric == 'entropy' and values.min() < 0:
            issues.append("Negative entropy values")
        if metric == 'coherence' and (values.min() < -1 or values.max() > 1):
            issues.append("Coherence outside [-1,1] range")
        if np.isinf(values).any():
            issues.append("Infinite values detected")
        
        kec_stats[metric]['issues'] = issues
    
    # Display results
    for metric, stats in kec_stats.items():
        print(f"\n  {metric.upper()}:")
        print(f"    Valid: {stats['count']:,} | Missing: {stats['missing']:,}")
        print(f"    Range: [{stats['min']:.3f}, {stats['max']:.3f}]")
        print(f"    Mean±SD: {stats['mean']:.3f}±{stats['std']:.3f}")
        if stats['issues']:
            print(f"    ⚠️ Issues: {', '.join(stats['issues'])}")
    
    # Save KEC audit
    with open(AUDIT_OUT / 'kec_audit.json', 'w') as f:
        json.dump(kec_stats, f, indent=2, default=str)
    
else:
    print("❌ KEC metrics not available for audit")

## 3. Data Integration Quality Assessment

In [ ]:
# Integration audit
if not datasets['merged'].empty:
    merged_df = datasets['merged']
    
    print(f"📊 Integration Analysis:")
    print(f"  Total rows: {len(merged_df):,}")
    print(f"  Total columns: {len(merged_df.columns)}")
    
    # Key columns check
    key_columns = {
        'token_norm': 'token_norm' in merged_df.columns,
        'entropy': 'entropy' in merged_df.columns,
        'curvature': 'curvature' in merged_df.columns,
        'coherence': 'coherence' in merged_df.columns,
        'reading_measures': any(col in merged_df.columns for col in ['TRT', 'FFD', 'GD', 'GPT'])
    }
    
    print(f"\n  Key columns present:")
    for col, present in key_columns.items():
        status = "✅" if present else "❌"
        print(f"    {status} {col}")
    
    # KEC coverage analysis
    kec_cols = [col for col in ['entropy', 'curvature', 'coherence'] if col in merged_df.columns]
    if kec_cols:
        print(f"\n  KEC Coverage:")
        for col in kec_cols:
            coverage = merged_df[col].notna().sum() / len(merged_df) * 100
            print(f"    {col}: {coverage:.1f}%")
        
        # Complete KEC triplet
        complete_kec = merged_df[kec_cols].notna().all(axis=1).sum()
        complete_pct = complete_kec / len(merged_df) * 100
        print(f"    Complete KEC triplet: {complete_pct:.1f}% ({complete_kec:,} cases)")
    
    # Token normalization check
    if 'token_norm' in merged_df.columns:
        unique_tokens = merged_df['token_norm'].nunique()
        null_tokens = merged_df['token_norm'].isna().sum()
        print(f"\n  Token Analysis:")
        print(f"    Unique tokens: {unique_tokens:,}")
        print(f"    Null tokens: {null_tokens:,}")
    
    # Save integration audit
    integration_stats = {
        'total_rows': len(merged_df),
        'key_columns': key_columns,
        'kec_coverage': {col: float(merged_df[col].notna().sum() / len(merged_df) * 100) 
                        for col in kec_cols} if kec_cols else {},
        'complete_kec_pct': float(complete_pct) if kec_cols else 0
    }
    
    with open(AUDIT_OUT / 'integration_audit.json', 'w') as f:
        json.dump(integration_stats, f, indent=2)
    
else:
    print("❌ Merged data not available for audit")

## 4. Statistical Methods Validation

In [ ]:
# Statistical results audit
if not datasets['coeffs'].empty:
    coeffs_df = datasets['coeffs']
    
    print(f"📊 Statistical Results Analysis:")
    print(f"  Total coefficients: {len(coeffs_df)}")
    
    # Normalize column names
    cols_map = {}
    for col in coeffs_df.columns:
        if 'response' in col.lower() or 'outcome' in col.lower():
            cols_map[col] = 'response'
        elif 'term' in col.lower() or 'predictor' in col.lower():
            cols_map[col] = 'term'
        elif 'coef' in col.lower() or 'beta' in col.lower() or 'estimate' in col.lower():
            cols_map[col] = 'coef'
        elif col.lower() in ['p', 'pval', 'p_value', 'pvalue']:
            cols_map[col] = 'pval'
    
    if cols_map:
        coeffs_df = coeffs_df.rename(columns=cols_map)
    
    # Check required columns
    required_cols = ['response', 'term', 'coef', 'pval']
    available_stat_cols = [col for col in required_cols if col in coeffs_df.columns]
    
    print(f"  Available columns: {available_stat_cols}")
    
    if len(available_stat_cols) >= 3:  # At least response, term, and one of coef/pval
        # Identify KEC effects
        if 'term' in coeffs_df.columns:
            kec_terms = coeffs_df['term'].str.contains(r'\b(entropy|curvature|coherence)\b', 
                                                      case=False, regex=True, na=False)
            kec_effects = coeffs_df[kec_terms]
            
            print(f"  KEC effects found: {len(kec_effects)}")
            
            if len(kec_effects) > 0 and 'response' in coeffs_df.columns:
                # Analyze by response type
                response_types = kec_effects['response'].value_counts()
                print(f"\n  Effects by response:")
                for response, count in response_types.items():
                    print(f"    {response}: {count}")
                
                # Significance analysis
                if 'pval' in coeffs_df.columns:
                    sig_effects = kec_effects[kec_effects['pval'] < 0.05]
                    print(f"\n  Significant effects (p<0.05): {len(sig_effects)}")
                    
                    if len(sig_effects) > 0:
                        print(f"  Top significant effects:")
                        top_effects = sig_effects.nsmallest(5, 'pval')[['response', 'term', 'pval']]
                        for _, row in top_effects.iterrows():
                            print(f"    {row['response']} ~ {row['term']}: p={row['pval']:.4f}")
        
        # FDR correction validation
        if 'pval' in coeffs_df.columns and len(kec_effects) > 0:
            print(f"\n  FDR Correction Validation:")
            
            # Apply BH correction
            if 'response' in coeffs_df.columns:
                fdr_results = []
                for response in kec_effects['response'].unique():
                    response_effects = kec_effects[kec_effects['response'] == response]
                    if len(response_effects) > 0:
                        _, q_vals, _, _ = multipletests(response_effects['pval'], 
                                                       method='fdr_bh', alpha=0.05)
                        sig_fdr = (q_vals < 0.05).sum()
                        fdr_results.append({'response': response, 'sig_fdr': sig_fdr, 
                                          'total': len(response_effects)})
                
                for result in fdr_results:
                    print(f"    {result['response']}: {result['sig_fdr']}/{result['total']} significant after FDR")
    
    # Save statistical audit
    stat_audit = {
        'total_coefficients': len(coeffs_df),
        'available_columns': list(coeffs_df.columns),
        'kec_effects_count': len(kec_effects) if 'kec_effects' in locals() else 0,
        'significant_effects': len(sig_effects) if 'sig_effects' in locals() else 0
    }
    
    with open(AUDIT_OUT / 'statistical_audit.json', 'w') as f:
        json.dump(stat_audit, f, indent=2)
    
else:
    print("❌ Statistical coefficients not available for audit")

## 5. H* Validation Framework Assessment

In [ ]:
# H* validation (simplified version of tools/validate_hstar.py)
def assess_h_star_criteria():
    """Assess H* validation criteria"""
    
    # Check pipeline prerequisites
    pipeline_ok = True
    required_files = [
        DATA_PROC / 'zuco_aligned.csv',
        DATA_PROC / 'kec' / 'metrics_en.csv'
    ]
    
    for file_path in required_files:
        if not file_path.exists():
            pipeline_ok = False
            break
    
    if not pipeline_ok:
        return {'decision': 'PIPELINE_BROKEN', 'reason': 'Required files missing'}
    
    if datasets['merged'].empty or datasets['coeffs'].empty:
        return {'decision': 'PIPELINE_BROKEN', 'reason': 'Processed data missing'}
    
    # Analyze coefficients for H* criteria
    coeffs_df = datasets['coeffs'].copy()
    
    # Normalize column names (simplified)
    col_mapping = {}
    for col in coeffs_df.columns:
        col_lower = col.lower()
        if 'response' in col_lower or 'outcome' in col_lower:
            col_mapping[col] = 'outcome'
        elif 'term' in col_lower:
            col_mapping[col] = 'term'
        elif 'coef' in col_lower or 'beta' in col_lower:
            col_mapping[col] = 'beta'
        elif col_lower in ['p', 'pval', 'p_value']:
            col_mapping[col] = 'pval'
    
    coeffs_df = coeffs_df.rename(columns=col_mapping)
    
    if not all(col in coeffs_df.columns for col in ['outcome', 'term', 'beta', 'pval']):
        return {'decision': 'PIPELINE_BROKEN', 'reason': 'Required coefficient columns missing'}
    
    # Identify KEC effects on reading measures
    is_kec = coeffs_df['term'].str.contains(r'\b(entropy|curvature|coherence)\b', 
                                           case=False, regex=True, na=False)
    is_reading = coeffs_df['outcome'].str.contains('ffd|gd|trt|gpt', case=False, na=False)
    
    kec_reading = coeffs_df[is_kec & is_reading].copy()
    
    if len(kec_reading) == 0:
        return {'decision': 'NOT_SUPPORTED', 'reason': 'No KEC effects on reading measures'}
    
    # Apply FDR correction
    _, q_vals, _, _ = multipletests(kec_reading['pval'], method='fdr_bh', alpha=0.05)
    kec_reading['qval'] = q_vals
    
    # Count significant positive effects
    sig_positive = ((kec_reading['qval'] < 0.05) & (kec_reading['beta'] > 0)).sum()
    
    # H* decision criteria (simplified)
    if sig_positive >= 2:
        decision = 'SUPPORTED'
        reason = f'{sig_positive} significant positive KEC effects on reading'
    elif sig_positive >= 1:
        decision = 'PARTIAL'
        reason = f'{sig_positive} significant positive KEC effect on reading'
    else:
        decision = 'NOT_SUPPORTED'
        reason = 'No significant positive KEC effects on reading'
    
    return {
        'decision': decision,
        'reason': reason,
        'kec_effects_total': len(kec_reading),
        'significant_positive': int(sig_positive),
        'top_effects': kec_reading.nsmallest(3, 'qval')[['outcome', 'term', 'beta', 'pval', 'qval']].to_dict('records')
    }

# Run H* assessment
print("📊 H* Validation Assessment:")
h_star_result = assess_h_star_criteria()

print(f"  Decision: {h_star_result['decision']}")
print(f"  Reason: {h_star_result['reason']}")

if 'kec_effects_total' in h_star_result:
    print(f"  Total KEC effects: {h_star_result['kec_effects_total']}")
    print(f"  Significant positive: {h_star_result['significant_positive']}")
    
    if h_star_result['top_effects']:
        print(f"\n  Top effects:")
        for effect in h_star_result['top_effects']:
            print(f"    {effect['outcome']} ~ {effect['term']}: β={effect['beta']:.3f}, q={effect['qval']:.4f}")

# Save H* assessment
with open(AUDIT_OUT / 'h_star_assessment.json', 'w') as f:
    json.dump(h_star_result, f, indent=2, default=str)

## 6. Summary and Recommendations

In [ ]:
# Generate comprehensive audit summary
audit_summary = {
    'timestamp': datetime.now().isoformat(),
    'pipeline_status': pipeline_status,
    'datasets_available': available,
    'h_star_decision': h_star_result['decision'] if 'h_star_result' in locals() else 'UNKNOWN'
}

# Collect issues and recommendations
issues = []
recommendations = []

if pipeline_status != 'COMPLETE':
    issues.append(f"Pipeline incomplete: {pipeline_status}")
    recommendations.append("Run complete data processing pipeline")

if 'kec_stats' in locals():
    for metric, stats in kec_stats.items():
        if stats['issues']:
            issues.extend([f"{metric}: {issue}" for issue in stats['issues']])
            recommendations.append(f"Investigate {metric} computation issues")

if datasets['merged'].empty:
    issues.append("Merged dataset not available")
    recommendations.append("Run data integration pipeline")

audit_summary['issues'] = issues
audit_summary['recommendations'] = recommendations

print("\n" + "="*60)
print("AUDIT SUMMARY")
print("="*60)

print(f"\n📊 Overall Status: {pipeline_status}")
print(f"📊 H* Decision: {audit_summary['h_star_decision']}")
print(f"📊 Available Datasets: {len(available)}/4")

if issues:
    print(f"\n⚠️ Issues Identified ({len(issues)}):")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")

if recommendations:
    print(f"\n💡 Recommendations ({len(recommendations)}):")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

print(f"\n📁 Audit outputs saved to: {AUDIT_OUT}")
print(f"📄 Detailed report: reports/SWOW_KEC_STATISTICAL_AUDIT_REPORT.md")

# Save final summary
with open(AUDIT_OUT / 'audit_summary.json', 'w') as f:
    json.dump(audit_summary, f, indent=2)

print("\n✅ Audit completed successfully!")